# Lean Statement Diversity Dataset — Perturbation Pipeline
Rule-based transforms — no API key required. Produces `(anchor, variant, transformation_type)` triples.

In [ ]:
import re, json, math
import pandas as pd
from pathlib import Path
from huggingface_hub import HfFileSystem
from itertools import permutations
import random
 
fs = HfFileSystem()
 
OUTPUT_FILE = Path("perturbation_pairs.jsonl")
print("Ready.")

Ready.


In [24]:
df = pd.read_json(
    "hf://datasets/FrenzyMath/mathlib_informal_v4.19.0/data.jsonl",
    lines=True,
)
df = df[0:10]
df.head()

,module_name,kind,name,start,stop,signature,type,value,docstring,informal_name,informal_description,index
0,"[Mathlib, CategoryTheory, Monoidal, Hopf_]",theorem,"[Hopf_, antipode_comul₂]",5768,9754,(A : Hopf_ C) :\n A.X.comul.hom ≫\n A.X...,∀ {C : Type u₁} [inst : CategoryTheory.Categor...,:= by\n -- We should write a version of `slic...,Auxiliary calculation for `antipode_comul`.\nT...,Antipode Antihomomorphism Condition for Comult...,For any Hopf monoid $A$ in a braided monoidal ...,14
1,"[Mathlib, CategoryTheory, Limits, Shapes, Equa...",theorem,"[CategoryTheory, Limits, coequalizer, existsUn...",36914,37120,{W : C} (k : Y ⟶ W) (h : f ≫ k = g ≫ k) : ∃! ...,∀ {C : Type u} [inst : CategoryTheory.Category...,:=\n Cofork.IsColimit.existsUnique (colimit.i...,NaN,Universal Property of Coequalizer (Unique Exis...,"Given two parallel morphisms $f, g : X \to Y$ ...",132
2,"[Mathlib, CategoryTheory, Limits, Shapes, Equa...",theorem,"[CategoryTheory, Limits, Fork, ι_ofι]",11746,11859,{P : C} (ι : P ⟶ X) (w : ι ≫ f = ι ≫ g) : (Fo...,∀ {C : Type u} [inst : CategoryTheory.Category...,:=\n rfl,NaN,Inclusion Morphism of Constructed Fork Equals ...,Given an object $P$ in a category $\mathcal{C}...,46
3,"[Mathlib, CategoryTheory, Limits, Shapes, Equa...",definition,"[CategoryTheory, Limits, ForkOfι, ext]",25812,26092,{P : C} {ι ι' : P ⟶ X} (w : ι ≫ f = ι ≫ g) (w...,{C : Type u} →\n [inst : CategoryTheory.Categ...,:=\n Fork.ext (Iso.refl _) (by simp [h]),Two forks of the form `ofι` are isomorphic whe...,Isomorphism of forks from equal morphisms,Given an object $P$ in a category $\mathcal{C}...,85
4,"[Mathlib, CategoryTheory, Limits, Shapes, Equa...",theorem,"[CategoryTheory, Limits, parallelPair_obj_one]",6710,6802,(f g : X ⟶ Y) : (parallelPair f g).obj one = Y,∀ {C : Type u} [inst : CategoryTheory.Category...,:= rfl,NaN,Parallel Pair Functor Maps One to Codomain,"For any two parallel morphisms $f, g : X \to Y...",24


In [25]:
df = df[['signature', 'type']]
df.head()

,signature,type
0,(A : Hopf_ C) :\n A.X.comul.hom ≫\n A.X...,∀ {C : Type u₁} [inst : CategoryTheory.Categor...
1,{W : C} (k : Y ⟶ W) (h : f ≫ k = g ≫ k) : ∃! ...,∀ {C : Type u} [inst : CategoryTheory.Category...
2,{P : C} (ι : P ⟶ X) (w : ι ≫ f = ι ≫ g) : (Fo...,∀ {C : Type u} [inst : CategoryTheory.Category...
3,{P : C} {ι ι' : P ⟶ X} (w : ι ≫ f = ι ≫ g) (w...,{C : Type u} →\n [inst : CategoryTheory.Categ...
4,(f g : X ⟶ Y) : (parallelPair f g).obj one = Y,∀ {C : Type u} [inst : CategoryTheory.Category...


In [ ]:
import subprocess, os, re

PROJECT_DIR = os.path.abspath(".")
_TMP_LEAN = os.path.join(PROJECT_DIR, "_tmp_nb.lean")

def _run_lean(code: str) -> str:
    try:
        with open(_TMP_LEAN, "w") as f:
            f.write(code)
        result = subprocess.run(
            ["lake", "env", "lean", _TMP_LEAN],
            cwd=PROJECT_DIR,
            capture_output=True,
            text=True,
            timeout=300,
        )
        return result.stdout + result.stderr
    finally:
        if os.path.exists(_TMP_LEAN):
            os.remove(_TMP_LEAN)

def compile_lean(sig: str, type_str: str) -> bool:
    output = _run_lean(f"import Mathlib\n\nexample : {type_str} := by sorry\n")
    return "error:" not in output

def contrapositive_theorem(sig: str, type_str: str) -> str | None:
    output = _run_lean(
        f"import Mathlib\n\ncont"
    )
def negate_theorem(sig: str, type_str: str) -> str | None:
    output = _run_lean(
        f"import Negate\n\nexample : {type_str} := by\n  negate_state\n  extract_goal\n  sorry\n"
    )
    m = re.search(r"^theorem .*extracted.*$", output, re.MULTILINE)
    return m.group(0) if m else None

In [ ]:
def apply_perturbation_chains(
    df: pd.DataFrame,
    transforms: dict,
    n_permutations: int = 6,
    random_seed: int = 42,
) -> pd.DataFrame:
    rng = random.Random(random_seed)
    transform_names = list(transforms.keys())
    all_permutations = list(permutations(transform_names))

    results = []

    for _, row in df.iterrows():
        anchor_sig  = row["signature"]
        anchor_type = row["type"]

        k = min(n_permutations, len(all_permutations))
        sampled_permutations = rng.sample(all_permutations, k)

        for perm in sampled_permutations:
            current_sig  = anchor_sig
            current_type = anchor_type
            applied_so_far = []

            for transform_name in perm:
                fn = transforms[transform_name]

                result = fn(current_sig, current_type)
                if result is None:
                    break

                variant_sig, variant_type = result
                if variant_sig.strip() == current_sig.strip() and variant_type.strip() == current_type.strip():
                    break

                if not compile_lean(variant_sig, variant_type):
                    break

                # ✅ Compiled — save this intermediate as a datapoint
                applied_so_far.append(transform_name)
                results.append({
                    **{k: v for k, v in row.items() if k not in ("signature", "type")},
                    "anchor_signature":      anchor_sig,
                    "anchor_type":           anchor_type,
                    "variant_signature":     variant_sig,
                    "variant_type":          variant_type,
                    "perturbations_applied": list(applied_so_far),
                    "chain_depth":           len(applied_so_far),
                    "is_true":               is_true(list(applied_so_far), anchor_sig, anchor_type),
                })

                current_sig  = variant_sig
                current_type = variant_type

    result_df = pd.DataFrame(results)
    print(f"Generated {len(result_df):,} compiled variants from {len(df):,} anchors")
    if len(result_df):
        print(result_df["chain_depth"].value_counts().sort_index().to_string())
    return result_df